# Сравнение стратегий чанкирования

В этом ноутбуке проверяется retrieval-часть RAG: какие фрагменты документов возвращаются при разных способах разбиения текста.

LLM здесь не вызывается. Сравнение идёт на уровне поиска контекста: документы разбиваются на чанки, чанки сохраняются в Chroma, затем по вопросам извлекаются top-4 наиболее близких фрагмента.

Проверяются четыре класса из `src/chunkers.py`:

- `FixedChunker`
- `NltkSemanticChunker`
- `SemanticChunker`
- `RecursiveChunker`

## 1. Импорты

Подключаются классы чанкирования из проекта, загрузчик корпуса и инструменты для Chroma DB.

In [1]:
import os
import ast
import time
import shutil
import string
from pathlib import Path

import numpy as np
import pandas as pd

os.environ["TOKENIZERS_PARALLELISM"] = "false"

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from src.utils import CorpusLoader
from src.chunkers import (
    FixedChunker,
    NltkSemanticChunker,
    SemanticChunker,
    RecursiveChunker,
)

c:\Users\user\Desktop\3 курс\RAG\rag_chunking\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


## 1. Настройки эксперимента

В эксперименте сравниваются четыре стратегии чанкирования при одинаковых условиях.

Фиксируются:

- размер чанка;
- отсутствие overlap;
- embedding-модель;
- количество возвращаемых чанков;
- один и тот же корпус документов;
- один и тот же набор вопросов.

Меняется только класс chunker.

In [5]:
chunk_size = 512
chunk_overlap = 0
top_k = 4
n_questions = 100

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

results_dir = Path("results")
db_root = Path("data/retrieval_eval_dbs")

results_dir.mkdir(exist_ok=True)
db_root.mkdir(parents=True, exist_ok=True)

## 2. Загрузка данных

В проекте используются два файла:

- `data/corpus.json` — корпус документов, по которому выполняется поиск;
- `data/MultiHopRAG.json` — вопросы и эталонная информация для оценки retrieval.

Дальше один и тот же корпус будет разбит четырьмя разными стратегиями чанкирования.

In [6]:
documents = CorpusLoader().load_content()
questions_df = pd.read_json("data/MultiHopRAG.json")

print("documents:", len(documents))
print("questions:", len(questions_df))
print("columns:", list(questions_df.columns))

questions_df.head()

Successfully loaded content from corpus
documents: 609
questions: 2556
columns: ['query', 'answer', 'question_type', 'evidence_list']


,query,answer,question_type,evidence_list
0,Who is the individual associated with the cryp...,Sam Bankman-Fried,inference_query,[{'title': 'The FTX trial is bigger than Sam B...
1,Which individual is implicated in both inflati...,Donald Trump,inference_query,[{'title': 'Donald Trump defrauded banks with ...
2,Who is the figure associated with generative A...,Sam Altman,inference_query,[{'title': 'OpenAI's ex-chairman accuses board...
3,Do the TechCrunch article on software companie...,Yes,comparison_query,"[{'title': 'Here’s how Rainforest, a budding S..."
4,Which online betting platform provides a welco...,Caesars Sportsbook,inference_query,[{'title': '2023 Kentucky online sports bettin...


## 3. Формирование выборки для эксперимента

Для сравнения стратегий используется весь корпус документов и первые 100 вопросов из `MultiHopRAG.json`.

Такой формат соответствует логике основного `qa_pipeline`: разные chunker-классы проверяются на одинаковых документах и одинаковых вопросах.

In [7]:
eval_documents = documents
eval_questions_df = questions_df.iloc[:n_questions].reset_index(drop=True)

print("eval documents:", len(eval_documents))
print("eval questions:", len(eval_questions_df))

eval_questions_df[["query", "answer", "question_type"]].head()

eval documents: 609
eval questions: 100


,query,answer,question_type
0,Who is the individual associated with the cryp...,Sam Bankman-Fried,inference_query
1,Which individual is implicated in both inflati...,Donald Trump,inference_query
2,Who is the figure associated with generative A...,Sam Altman,inference_query
3,Do the TechCrunch article on software companie...,Yes,comparison_query
4,Which online betting platform provides a welco...,Caesars Sportsbook,inference_query


## 4. Проверка SemanticChunker на простом примере

Перед запуском на основном корпусе проверяется логика  `SemanticChunker`.

В тестовом тексте специально смешаны три разные темы:

- математический анализ;
- глубокое обучение;
- финансы.

Если чанкер работает корректно, он должен отделить смысловые блоки друг от друга.

In [8]:
demo_text = """
A derivative describes how a function changes when its argument changes. 
In calculus, derivatives are used to study growth, local extrema and the shape of a graph.
An integral can be interpreted as the area under a curve or as the accumulated value of a changing quantity.

Deep learning models consist of layers with trainable parameters. 
During training, backpropagation computes gradients of the loss function with respect to these parameters.
The optimizer then updates the weights to reduce the error on the training data.

Financial markets react to interest rates, inflation and expectations about future earnings.
Investors compare risk and return before choosing assets for a portfolio.
A bond price can change when market interest rates increase or decrease.
"""

demo_chunker = SemanticChunker(
    chunk_size=500,
    chunk_overlap=0,
    language="english",
)

demo_chunks = demo_chunker.split_text(demo_text)

# Проверяем, как текст разделился на смысловые чанки
for i, chunk in enumerate(demo_chunks, 1):
    print(f"chunk {i}:")
    print(chunk)
    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 779.38it/s]


chunk 1:
A derivative describes how a function changes when its argument changes. In calculus, derivatives are used to study growth, local extrema and the shape of a graph. An integral can be interpreted as the area under a curve or as the accumulated value of a changing quantity.

chunk 2:
Deep learning models consist of layers with trainable parameters. During training, backpropagation computes gradients of the loss function with respect to these parameters. The optimizer then updates the weights to reduce the error on the training data.

chunk 3:
Financial markets react to interest rates, inflation and expectations about future earnings. Investors compare risk and return before choosing assets for a portfolio. A bond price can change when market interest rates increase or decrease.



## 5. Стратегии чанкирования

Дальше задаются четыре стратегии из `src/chunkers.py`.

Все стратегии запускаются при одинаковых параметрах:

- `chunk_size = 512`;
- `chunk_overlap = 0`;
- язык для sentence-based стратегий — английский.

Так сравнивается именно способ разбиения текста, а не разные настройки pipeline

In [9]:
chunker_map = {
    "fixed": FixedChunker(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    ),
    "nltk_sentence": NltkSemanticChunker(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        language="english",
    ),
    "semantic": SemanticChunker(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        language="english",
        breakpoint_percentile_threshold=80,
    ),
    "recursive": RecursiveChunker(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    ),
}

# Проверяем список стратегий
list(chunker_map.keys())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1142.31it/s]


['fixed', 'nltk_sentence', 'semantic', 'recursive']

## 6. Разбиение документов на чанки

Для каждой стратегии документы разбиваются на чанки.

Дополнительно сохраняются технические характеристики:

- количество чанков;
- средняя длина чанка;
- минимальная и максимальная длина;
- время чанкирования.

Эти показатели нужны, чтобы сравнивать стратегии не только по качеству поиска, но и по размеру получившейся базы

In [10]:
def make_chunks(name, chunker, docs):
    start = time.time()
    chunks = chunker.split_documents(docs)
    elapsed = time.time() - start

    lengths = [len(item.page_content) for item in chunks]

    stats = {
        "strategy": name,
        "chunks_count": len(chunks),
        "avg_chunk_len": float(np.mean(lengths)) if lengths else 0,
        "median_chunk_len": float(np.median(lengths)) if lengths else 0,
        "min_chunk_len": int(np.min(lengths)) if lengths else 0,
        "max_chunk_len": int(np.max(lengths)) if lengths else 0,
        "chunking_time_sec": round(elapsed, 3),
    }

    return chunks, stats

## 7. Embedding-модель и Chroma DB

После чанкирования каждый фрагмент текста переводится в embedding.

Embedding — это числовое представление текста, которое позволяет искать похожие фрагменты по смыслу, а не только по совпадению слов.

Для хранения embeddings используется Chroma DB. Для каждой стратегии создаётся отдельная база, чтобы результаты не смешивались между собой.

In [11]:
embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 767.69it/s]


## 8. Построение Chroma DB и поиск чанков

Для каждой стратегии создаётся отдельная Chroma-база.

Дальше по каждому вопросу выполняется поиск `top-4` наиболее близких чанков.  
В результат сохраняются:

- название стратегии;
- номер вопроса;
- текст вопроса;
- найденные чанки;
- metadata найденных документов.

In [12]:
def retrieve_chunks(name, chunks, queries, k=4):
    db_dir = db_root / name

    if db_dir.exists():
        shutil.rmtree(db_dir)

    index_start = time.time()

    db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=str(db_dir),
    )

    index_time = time.time() - index_start

    retriever = db.as_retriever(search_kwargs={"k": k})

    rows = []
    retrieval_start = time.time()

    for question_id, query in enumerate(queries):
        found_docs = retriever.invoke(query)

        rows.append({
            "strategy": name,
            "question_id": question_id,
            "query": query,
            "retrieved_chunks": [doc.page_content for doc in found_docs],
            "retrieved_metadata": [doc.metadata for doc in found_docs],
        })

    retrieval_time = time.time() - retrieval_start

    timing = {
        "strategy": name,
        "index_time_sec": round(index_time, 3),
        "retrieval_time_sec": round(retrieval_time, 3),
    }

    return pd.DataFrame(rows), timing

## 9. Запуск эксперимента

На этом этапе один и тот же корпус документов и один и тот же набор вопросов проходят через четыре стратегии чанкирования.

Для каждой стратегии выполняются шаги:

1. документы разбиваются на чанки;
2. чанки сохраняются в отдельную Chroma-базу;
3. по каждому вопросу извлекаются top-4 наиболее релевантных чанка;
4. сохраняются результаты и техническая статистика.


In [14]:
# Сохраняем то, что уже успело посчитаться до остановки большой ячейки

if retrieval_parts:
    retrieval_df = pd.concat(retrieval_parts, ignore_index=True)
    chunk_stats_df = pd.DataFrame(chunk_stats_rows)
    timing_df = pd.DataFrame(timing_rows)

    retrieval_df.to_csv(results_dir / "chunkers_retrieval_results_partial.csv", index=False)
    chunk_stats_df.to_csv(results_dir / "chunkers_chunk_stats_partial.csv", index=False)
    timing_df.to_csv(results_dir / "chunkers_time_stats_partial.csv", index=False)

    display(chunk_stats_df)
    display(timing_df)
else:
    print("Пока нет сохранённых результатов")

,strategy,chunks_count,avg_chunk_len,median_chunk_len,min_chunk_len,max_chunk_len,chunking_time_sec
0,fixed,12601,499.396556,512.0,1,512,41.928


,strategy,index_time_sec,retrieval_time_sec
0,fixed,2395.32,19.988


In [15]:
def save_current_results(suffix="full"):
    retrieval_df = pd.concat(retrieval_parts, ignore_index=True)
    chunk_stats_df = pd.DataFrame(chunk_stats_rows)
    timing_df = pd.DataFrame(timing_rows)

    retrieval_df.to_csv(results_dir / f"chunkers_retrieval_results_{suffix}.csv", index=False)
    chunk_stats_df.to_csv(results_dir / f"chunkers_chunk_stats_{suffix}.csv", index=False)
    timing_df.to_csv(results_dir / f"chunkers_time_stats_{suffix}.csv", index=False)

    return retrieval_df, chunk_stats_df, timing_df


def run_strategy(name):
    print("strategy:", name)

    chunker = chunker_map[name]

    chunks, chunk_stats = make_chunks(name, chunker, eval_documents)
    results_part, timing = retrieve_chunks(name, chunks, queries, k=top_k)

    results_part.to_csv(results_dir / f"{name}_retrieval_results_full.csv", index=False)

    retrieval_parts.append(results_part)
    chunk_stats_rows.append(chunk_stats)
    timing_rows.append(timing)

    retrieval_df, chunk_stats_df, timing_df = save_current_results(suffix="full")

    print(chunk_stats)
    print(timing)

    return retrieval_df, chunk_stats_df, timing_df

## 11. Запуск NltkSemanticChunker

`NltkSemanticChunker` сначала делит текст на предложения через NLTK, а затем объединяет предложения в чанки заданного размера.

In [17]:
import gc

run_id = int(time.time())

def retrieve_chunks_safe(name, chunks, queries, k=4):
    # Создаём новую папку, чтобы не удалять старую Chroma-базу
    db_dir = db_root / f"{name}_{run_id}"

    index_start = time.time()

    db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=str(db_dir),
    )

    index_time = time.time() - index_start

    retriever = db.as_retriever(search_kwargs={"k": k})

    rows = []
    retrieval_start = time.time()

    for question_id, query in enumerate(queries):
        found_docs = retriever.invoke(query)

        rows.append({
            "strategy": name,
            "question_id": question_id,
            "query": query,
            "retrieved_chunks": [doc.page_content for doc in found_docs],
            "retrieved_metadata": [doc.metadata for doc in found_docs],
        })

    retrieval_time = time.time() - retrieval_start

    timing = {
        "strategy": name,
        "index_time_sec": round(index_time, 3),
        "retrieval_time_sec": round(retrieval_time, 3),
    }

    # Освобождаем объекты Chroma, чтобы Windows не держал файлы открытыми
    del retriever
    del db
    gc.collect()

    return pd.DataFrame(rows), timing


def run_strategy_safe(name):
    print("strategy:", name)

    chunker = chunker_map[name]

    chunks, chunk_stats = make_chunks(name, chunker, eval_documents)
    results_part, timing = retrieve_chunks_safe(name, chunks, queries, k=top_k)

    results_part.to_csv(results_dir / f"{name}_retrieval_results_full.csv", index=False)

    retrieval_parts.append(results_part)
    chunk_stats_rows.append(chunk_stats)
    timing_rows.append(timing)

    retrieval_df, chunk_stats_df, timing_df = save_current_results(suffix="full")

    print(chunk_stats)
    print(timing)

    return retrieval_df, chunk_stats_df, timing_df

In [18]:
retrieval_df, chunk_stats_df, timing_df = run_strategy_safe("nltk_sentence")

display(chunk_stats_df)
display(timing_df)

strategy: nltk_sentence
{'strategy': 'nltk_sentence', 'chunks_count': 14588, 'avg_chunk_len': 429.33575541540995, 'median_chunk_len': 447.0, 'min_chunk_len': 3, 'max_chunk_len': 1700, 'chunking_time_sec': 6.82}
{'strategy': 'nltk_sentence', 'index_time_sec': 1505.869, 'retrieval_time_sec': 9.611}


,strategy,chunks_count,avg_chunk_len,median_chunk_len,min_chunk_len,max_chunk_len,chunking_time_sec
0,fixed,12601,499.396556,512.0,1,512,41.928
1,nltk_sentence,14588,429.335755,447.0,3,1700,6.820


,strategy,index_time_sec,retrieval_time_sec
0,fixed,2395.320,19.988
1,nltk_sentence,1505.869,9.611


## 13. Запуск SemanticChunker

`SemanticChunker` строит разбиение по смысловым переходам между соседними предложениями.

Логика класса:

1. текст делится на предложения;
2. для предложений строятся embeddings;
3. между соседними предложениями считается cosine distance;
4. при высоком расстоянии создаётся новая смысловая граница;
5. полученные группы дополнительно проверяются по `chunk_size`.


In [19]:
retrieval_df, chunk_stats_df, timing_df = run_strategy_safe("semantic")

display(chunk_stats_df)
display(timing_df)

strategy: semantic
{'strategy': 'semantic', 'chunks_count': 19978, 'avg_chunk_len': 313.3354690159175, 'median_chunk_len': 358.0, 'min_chunk_len': 1, 'max_chunk_len': 512, 'chunking_time_sec': 2264.131}
{'strategy': 'semantic', 'index_time_sec': 1588.973, 'retrieval_time_sec': 8.243}


,strategy,chunks_count,avg_chunk_len,median_chunk_len,min_chunk_len,max_chunk_len,chunking_time_sec
0,fixed,12601,499.396556,512.0,1,512,41.928
1,nltk_sentence,14588,429.335755,447.0,3,1700,6.820
2,semantic,19978,313.335469,358.0,1,512,2264.131


,strategy,index_time_sec,retrieval_time_sec
0,fixed,2395.320,19.988
1,nltk_sentence,1505.869,9.611
2,semantic,1588.973,8.243


In [20]:
retrieval_df, chunk_stats_df, timing_df = run_strategy_safe("recursive")

display(chunk_stats_df)
display(timing_df)


strategy: recursive
{'strategy': 'recursive', 'chunks_count': 17415, 'avg_chunk_len': 359.80005742176286, 'median_chunk_len': 398.0, 'min_chunk_len': 3, 'max_chunk_len': 512, 'chunking_time_sec': 1.862}
{'strategy': 'recursive', 'index_time_sec': 1289.33, 'retrieval_time_sec': 8.586}


,strategy,chunks_count,avg_chunk_len,median_chunk_len,min_chunk_len,max_chunk_len,chunking_time_sec
0,fixed,12601,499.396556,512.0,1,512,41.928
1,nltk_sentence,14588,429.335755,447.0,3,1700,6.820
2,semantic,19978,313.335469,358.0,1,512,2264.131
3,recursive,17415,359.800057,398.0,3,512,1.862


,strategy,index_time_sec,retrieval_time_sec
0,fixed,2395.320,19.988
1,nltk_sentence,1505.869,9.611
2,semantic,1588.973,8.243
3,recursive,1289.330,8.586


## 14. Сохранение результатов по всем стратегиям

На этом этапе уже получены результаты для всех четырёх стратегий чанкирования.

Сохраняются три таблицы:

- найденные чанки по каждому вопросу;
- статистика по чанкам;
- время индексации и поиска.

In [21]:
retrieval_df, chunk_stats_df, timing_df = save_current_results(suffix="full")

display(chunk_stats_df)
display(timing_df)

print("retrieval rows:", len(retrieval_df))

,strategy,chunks_count,avg_chunk_len,median_chunk_len,min_chunk_len,max_chunk_len,chunking_time_sec
0,fixed,12601,499.396556,512.0,1,512,41.928
1,nltk_sentence,14588,429.335755,447.0,3,1700,6.820
2,semantic,19978,313.335469,358.0,1,512,2264.131
3,recursive,17415,359.800057,398.0,3,512,1.862


,strategy,index_time_sec,retrieval_time_sec
0,fixed,2395.320,19.988
1,nltk_sentence,1505.869,9.611
2,semantic,1588.973,8.243
3,recursive,1289.330,8.586


retrieval rows: 400


## 15. Подготовка функций для retrieval-метрик

Retrieval-метрики показывают, насколько хорошо стратегия чанкирования помогает находить релевантные фрагменты.

Найденные чанки сравниваются с `evidence_list` из датасета.

Используются три метрики:

- `Hits@4` — найден ли хотя бы один релевантный фрагмент среди top-4;
- `MAP@4` — насколько хорошо релевантные фрагменты ранжируются;
- `MRR@4` — насколько высоко расположен первый релевантный фрагмент.

In [22]:
def parse_value(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except Exception:
            return value
    return value


def collect_texts(value):
    value = parse_value(value)

    if value is None:
        return []

    if isinstance(value, str):
        return [value]

    if isinstance(value, dict):
        texts = []

        for key, item in value.items():
            if key.lower() in {"fact", "text", "content", "evidence", "sentence", "title", "source"}:
                texts.extend(collect_texts(item))
            elif isinstance(item, (list, dict)):
                texts.extend(collect_texts(item))

        return texts

    if isinstance(value, list):
        texts = []

        for item in value:
            texts.extend(collect_texts(item))

        return texts

    return []


def normalize_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(text.split())
    return text


def overlap_score(left, right):
    left_tokens = {token for token in normalize_text(left).split() if len(token) > 3}
    right_tokens = {token for token in normalize_text(right).split() if len(token) > 3}

    if not left_tokens or not right_tokens:
        return 0

    return len(left_tokens & right_tokens) / len(left_tokens)


def has_relevant_text(chunk, evidence_texts):
    chunk_norm = normalize_text(chunk)

    for evidence in evidence_texts:
        evidence_norm = normalize_text(evidence)

        if len(evidence_norm) >= 30 and evidence_norm in chunk_norm:
            return True

        if overlap_score(evidence, chunk) >= 0.6:
            return True

    return False

## 16. Расчёт retrieval-метрик

Для каждой стратегии проверяется, насколько хорошо top-4 найденных чанка совпадают с эталонными evidence-фрагментами.

Метрики считаются отдельно для каждой стратегии чанкирования.

In [23]:
def calculate_metrics(results, source_df, k=4):
    rows = []

    for name, group in results.groupby("strategy"):
        hits = []
        average_precisions = []
        reciprocal_ranks = []

        group = group.sort_values("question_id")

        for _, row in group.iterrows():
            question_id = int(row["question_id"])
            evidence_texts = collect_texts(source_df.iloc[question_id].get("evidence_list", []))

            retrieved_chunks = parse_value(row["retrieved_chunks"])

            if not isinstance(retrieved_chunks, list):
                retrieved_chunks = []

            relevance = [
                has_relevant_text(chunk, evidence_texts)
                for chunk in retrieved_chunks[:k]
            ]

            hits.append(int(any(relevance)))

            relevant_count = 0
            precisions = []

            for rank, is_relevant in enumerate(relevance, start=1):
                if is_relevant:
                    relevant_count += 1
                    precisions.append(relevant_count / rank)

            average_precisions.append(
                sum(precisions) / len(precisions) if precisions else 0
            )

            first_relevant_rank = 0

            for rank, is_relevant in enumerate(relevance, start=1):
                if is_relevant:
                    first_relevant_rank = rank
                    break

            reciprocal_ranks.append(
                1 / first_relevant_rank if first_relevant_rank else 0
            )

        rows.append({
            "strategy": name,
            f"Hits@{k}": float(np.mean(hits)),
            f"MAP@{k}": float(np.mean(average_precisions)),
            f"MRR@{k}": float(np.mean(reciprocal_ranks)),
            "questions_count": len(group),
        })

    return pd.DataFrame(rows).sort_values(f"Hits@{k}", ascending=False)


metrics_df = calculate_metrics(
    retrieval_df,
    eval_questions_df,
    k=top_k,
)

metrics_df

,strategy,Hits@4,MAP@4,MRR@4,questions_count
0,fixed,0.62,0.410278,0.436667,100
1,nltk_sentence,0.55,0.348056,0.360833,100
2,recursive,0.54,0.386389,0.400000,100
3,semantic,0.54,0.361389,0.367500,100


## 17. Итоговая таблица

In [24]:
summary_df = (
    metrics_df
    .merge(chunk_stats_df, on="strategy", how="left")
    .merge(timing_df, on="strategy", how="left")
)

summary_df = summary_df.sort_values("Hits@4", ascending=False)

summary_path = results_dir / "chunkers_retrieval_summary_full.csv"
summary_df.to_csv(summary_path, index=False)

summary_df

,strategy,Hits@4,MAP@4,MRR@4,questions_count,chunks_count,avg_chunk_len,median_chunk_len,min_chunk_len,max_chunk_len,chunking_time_sec,index_time_sec,retrieval_time_sec
0,fixed,0.62,0.410278,0.436667,100,12601,499.396556,512.0,1,512,41.928,2395.320,19.988
1,nltk_sentence,0.55,0.348056,0.360833,100,14588,429.335755,447.0,3,1700,6.820,1505.869,9.611
2,recursive,0.54,0.386389,0.400000,100,17415,359.800057,398.0,3,512,1.862,1289.330,8.586
3,semantic,0.54,0.361389,0.367500,100,19978,313.335469,358.0,1,512,2264.131,1588.973,8.243


## 19. Дополнительная строгая проверка retrieval-метрик


Отличие от предыдущей оценки:

- используется только поле `fact` из `evidence_list`;
- релевантность засчитывается только при дословном вхождении `fact` в найденный chunk;
- `null_query` исключаются из расчёта.


In [25]:
def get_evidence_facts(value):
    value = parse_value(value)

    if not isinstance(value, list):
        return []

    facts = []

    for item in value:
        if isinstance(item, dict) and "fact" in item:
            fact = item.get("fact")

            if fact:
                facts.append(str(fact))

    return facts


def compact_text(text):
    text = str(text).lower()
    text = text.replace(" ", "")
    text = text.replace("\n", "")
    text = text.replace("\t", "")
    return text


def matched_fact_indexes(chunk, facts):
    chunk_text = compact_text(chunk)
    matched = set()

    for i, fact in enumerate(facts):
        fact_text = compact_text(fact)

        if fact_text and fact_text in chunk_text:
            matched.add(i)

    return matched


def calculate_strict_metrics_like_partner(results, source_df, k=4):
    rows = []

    valid_question_ids = set(
        source_df.index[source_df["question_type"] != "null_query"]
    )

    results = results[results["question_id"].isin(valid_question_ids)].copy()

    for name, group in results.groupby("strategy"):
        hits = []
        average_precisions = []
        reciprocal_ranks = []

        group = group.sort_values("question_id")

        for _, row in group.iterrows():
            question_id = int(row["question_id"])
            facts = get_evidence_facts(source_df.loc[question_id, "evidence_list"])

            retrieved_chunks = parse_value(row["retrieved_chunks"])

            if not isinstance(retrieved_chunks, list):
                retrieved_chunks = []

            seen_facts = set()
            precision_values = []
            first_relevant_rank = 0

            for rank, chunk in enumerate(retrieved_chunks[:k], start=1):
                current_matches = matched_fact_indexes(chunk, facts)
                new_matches = current_matches - seen_facts

                if new_matches:
                    seen_facts.update(new_matches)
                    precision_values.append(len(seen_facts) / rank)

                    if first_relevant_rank == 0:
                        first_relevant_rank = rank

            hits.append(int(len(seen_facts) > 0))

            denominator = min(len(facts), k) if facts else 1
            average_precisions.append(sum(precision_values) / denominator)

            reciprocal_ranks.append(
                1 / first_relevant_rank if first_relevant_rank else 0
            )

        rows.append({
            "strategy": name,
            f"Hits@{k}": float(np.mean(hits)),
            f"MAP@{k}": float(np.mean(average_precisions)),
            f"MRR@{k}": float(np.mean(reciprocal_ranks)),
            "questions_count": len(group),
        })

    return pd.DataFrame(rows).sort_values(f"Hits@{k}", ascending=False)


strict_metrics_df = calculate_strict_metrics_like_partner(
    retrieval_df,
    eval_questions_df,
    k=top_k,
)

strict_metrics_df

,strategy,Hits@4,MAP@4,MRR@4,questions_count
3,semantic,0.373626,0.124161,0.246337,91
2,recursive,0.362637,0.112943,0.257326,91
1,nltk_sentence,0.340659,0.099741,0.212454,91
0,fixed,0.329670,0.085165,0.195055,91


In [26]:
strict_summary_df = (
    strict_metrics_df
    .merge(chunk_stats_df, on="strategy", how="left")
    .merge(timing_df, on="strategy", how="left")
)

strict_summary_df = strict_summary_df.sort_values("Hits@4", ascending=False)

strict_summary_path = results_dir / "chunkers_retrieval_summary_strict_100.csv"
strict_summary_df.to_csv(strict_summary_path, index=False)

strict_summary_df

,strategy,Hits@4,MAP@4,MRR@4,questions_count,chunks_count,avg_chunk_len,median_chunk_len,min_chunk_len,max_chunk_len,chunking_time_sec,index_time_sec,retrieval_time_sec
0,semantic,0.373626,0.124161,0.246337,91,19978,313.335469,358.0,1,512,2264.131,1588.973,8.243
1,recursive,0.362637,0.112943,0.257326,91,17415,359.800057,398.0,3,512,1.862,1289.330,8.586
2,nltk_sentence,0.340659,0.099741,0.212454,91,14588,429.335755,447.0,3,1700,6.820,1505.869,9.611
3,fixed,0.329670,0.085165,0.195055,91,12601,499.396556,512.0,1,512,41.928,2395.320,19.988
